# 03 — Model Training

## Why This Architecture Progression?

Class (Block 4) taught the **exact evolution** we implement here:

```
SimpleRNN  →  suffers vanishing gradient on long sequences
    ↓
GRU        →  2 gates, faster, fixes most vanishing gradient issues
    ↓
LSTM       →  4 gates + cell state highway, best long-range memory
    ↓
LSTM + Attention  →  learns WHICH timesteps matter most (our final model)
```

We train all 4 and compare RMSE — this becomes the architecture comparison slide.

### Why NOT a CNN?
Block 2 taught CNNs for spatial data (images — height × width × channels).  
Engine sensor data is **temporal**, not spatial. A CNN kernel sliding over pixels has no analog here — adjacent timesteps are not like adjacent pixels. LSTMs maintain a hidden state that accumulates sequence context; CNNs discard it.

### Why NOT a standard Dense ANN?
Block 1 taught that stacking Dense layers on non-sequential data loses ordering.  
Feeding 30 timesteps flattened into 510 numbers ([30 × 17]) treats cycle t=1 and t=30 as interchangeable — catastrophic for RUL prediction where recency matters.

---
**Run on Google Colab for free GPU — upload `data/processed/*.npy` first.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from dataclasses import dataclass

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

plt.style.use('dark_background')
PROC_DIR  = Path('../data/processed')
MODEL_DIR = Path('../backend/model')
MODEL_DIR.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {[g.name for g in tf.config.list_physical_devices("GPU")]}')

In [ ]:
# ── Load processed arrays (output of 02_preprocessing.ipynb) ──────────────
X_train = np.load(PROC_DIR / 'X_train.npy')  # [n_samples, 30, 17]
y_train = np.load(PROC_DIR / 'y_train.npy')
X_test  = np.load(PROC_DIR / 'X_test.npy')
y_test  = np.load(PROC_DIR / 'y_test.npy')

SEQ_LEN    = X_train.shape[1]   # 30 cycles lookback
N_FEATURES = X_train.shape[2]   # 17 (3 op settings + 14 sensors)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'Sequence length: {SEQ_LEN}, Features: {N_FEATURES}')

## Shared Training Config

All four models use identical hyperparameters so comparisons are fair.

**Huber loss vs MSE:** Block 1 taught MSE (RMSE). We use Huber (delta=10) because it behaves like MSE for small errors but like MAE for large ones — robust to the handful of engines with extreme RUL values. Same gradient descent mechanics, different loss surface shape.

In [ ]:
# Shared hyperparameters — identical across all 4 architectures
BATCH_SIZE = 64
EPOCHS     = 100
LR         = 1e-3
DROPOUT    = 0.2     # Block 2: forces neurons to learn independent features
L2         = 1e-4

def make_callbacks(model_name: str):
    """Block 1 / Block 4: EarlyStopping + ReduceLROnPlateau + ModelCheckpoint."""
    return [
        # EarlyStopping: stop when val_loss stops improving (patience=10 epochs)
        # Block 4 used patience=2 for retail — we need more for RUL regression
        callbacks.EarlyStopping(
            monitor='val_loss', patience=10,
            restore_best_weights=True, verbose=1
        ),
        # ReduceLROnPlateau: halve LR when stuck (Block 1: learning rate too large = bouncing)
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            MODEL_DIR / f'{model_name}.keras',
            monitor='val_loss', save_best_only=True, verbose=0
        ),
    ]

def train_model(model, model_name):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.Huber(delta=10.0),
        metrics=['mae']
    )
    history = model.fit(
        X_train, y_train,
        validation_split=0.2,   # 80/20 split — Block 1 golden rule
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,  # mini-batch GD — Block 1
        callbacks=make_callbacks(model_name),
        verbose=0
    )
    y_pred = model.predict(X_test, batch_size=128, verbose=0).flatten()
    rmse   = float(np.sqrt(np.mean((y_pred - y_test) ** 2)))
    mae    = float(np.mean(np.abs(y_pred - y_test)))
    print(f'{model_name:30s}  RMSE={rmse:.2f}  MAE={mae:.2f}  '
          f'epochs={len(history.history["loss"])}')
    return history, y_pred, rmse, mae

results = {}  # will hold history + metrics for all 4 models

## Model 1: SimpleRNN Baseline

**Block 4 taught:** SimpleRNN maintains a single hidden state `h_t = tanh(W_xh·x_t + W_hh·h_{t-1} + b_h)`.  
The tanh compresses context — after 30 time steps the signal from cycle t=1 is diluted 29 times.  
This is the **vanishing gradient problem**: early cycles barely influence the final prediction.  
For RUL prediction with a 30-cycle window this is a real problem — early degradation signals get lost.

We train it anyway to establish a baseline RMSE.

In [ ]:
def build_simple_rnn(seq_len, n_features):
    inp = keras.Input(shape=(seq_len, n_features))
    # SimpleRNN: single hidden state, tanh activation — Block 4 anatomy
    x = layers.SimpleRNN(128, return_sequences=True)(inp)
    x = layers.Dropout(DROPOUT)(x)                  # Block 2: prevent co-adaptation
    x = layers.SimpleRNN(64)(x)                     # second RNN layer, no return_sequences
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(L2))(x)
    x = layers.BatchNormalization()(x)               # Block 2: stabilizes training
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name='SimpleRNN')

model_rnn = build_simple_rnn(SEQ_LEN, N_FEATURES)
model_rnn.summary()
history_rnn, pred_rnn, rmse_rnn, mae_rnn = train_model(model_rnn, 'simple_rnn')
results['SimpleRNN'] = dict(history=history_rnn, rmse=rmse_rnn, mae=mae_rnn)

## Model 2: GRU

**Block 4 taught:** GRU (Cho et al., 2014) merges LSTM's forget + input gates into a single **update gate**.  
No separate cell state — the hidden state does both jobs.  
Fewer parameters than LSTM → trains faster. In practice, GRU and LSTM perform comparably.  

GRU gates:
- **Reset gate `r_t`:** How much past state to forget  
- **Update gate `z_t`:** How much new info vs old state to keep

In [ ]:
def build_gru(seq_len, n_features):
    inp = keras.Input(shape=(seq_len, n_features))
    # GRU: reset + update gates — Block 4. Fewer params than LSTM, comparable performance
    x = layers.GRU(128, return_sequences=True,
                   kernel_regularizer=keras.regularizers.l2(L2))(inp)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.GRU(64, kernel_regularizer=keras.regularizers.l2(L2))(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(L2))(x)
    x = layers.BatchNormalization()(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name='GRU')

model_gru = build_gru(SEQ_LEN, N_FEATURES)
model_gru.summary()
history_gru, pred_gru, rmse_gru, mae_gru = train_model(model_gru, 'gru')
results['GRU'] = dict(history=history_gru, rmse=rmse_gru, mae=mae_gru)

## Model 3: Stacked LSTM

**Block 4 taught:** LSTM (Hochreiter & Schmidhuber, 1997) adds a **cell state** — a second information channel (the "secure highway") that carries long-range context with minimal modification.

The 4 gates:
| Gate | Formula | Role |
|---|---|---|
| Forget | σ(W_f·[h,x] + b_f) | What to erase from cell state |
| Input | σ(W_i·[h,x] + b_i) | What new info to write |
| Cell candidate | tanh(W_c·[h,x] + b_c) | The candidate values |
| Output | σ(W_o·[h,x] + b_o) | What part of cell state to expose |

Update rule: `C_t = f_t * C_{t-1} + i_t * C̃_t`  ← erase old, write new  
Output: `h_t = o_t * tanh(C_t)`

In [ ]:
def build_lstm(seq_len, n_features):
    inp = keras.Input(shape=(seq_len, n_features))
    # LSTM: 4 gates + cell state highway — Block 4
    # return_sequences=True → pass full sequence to next LSTM layer
    x = layers.LSTM(128, return_sequences=True,
                    kernel_regularizer=keras.regularizers.l2(L2))(inp)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.LSTM(64, kernel_regularizer=keras.regularizers.l2(L2))(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(L2))(x)
    x = layers.BatchNormalization()(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name='LSTM')

model_lstm = build_lstm(SEQ_LEN, N_FEATURES)
model_lstm.summary()
history_lstm, pred_lstm, rmse_lstm, mae_lstm = train_model(model_lstm, 'lstm_base')
results['LSTM'] = dict(history=history_lstm, rmse=rmse_lstm, mae=mae_lstm)

## Model 4: LSTM + Dot-Product Attention (Final Model)

**Block 1/4 taught:** Attention was introduced as the mechanism that lets the model look at the **entire sequence at once** and weight each timestep by relevance — rather than compressing everything into a single fixed-size hidden state.

In this context: **not all 30 cycles are equally important for RUL prediction.**  
The most recent cycles (when degradation accelerates) should dominate. Attention learns this automatically.

Dot-product attention:
1. Score = (X · Xᵀ) / √d_model  ← how similar is each timestep to each other?
2. Weights = softmax(Score)      ← normalize to probability distribution (sums to 1)
3. Output = Weights · X          ← weighted sum of timesteps

The output of step 4 is visualized in notebook 04 to show WHICH cycles the model attends to.

In [ ]:
class DotProductAttention(layers.Layer):
    """
    Self-attention over the time dimension.
    Scores each timestep against all others to produce a weighted representation.
    Mirrors the attention mechanism from class (Block 1, Part 4).
    """
    def call(self, x):
        # x: [batch, seq_len, features]
        d = tf.cast(tf.shape(x)[-1], tf.float32)
        score   = tf.matmul(x, x, transpose_b=True) / tf.math.sqrt(d)  # [batch, T, T]
        weights = tf.nn.softmax(score, axis=-1)                          # attention probs
        attended = tf.matmul(weights, x)                                 # [batch, T, F]
        return attended, weights   # return weights so notebook 04 can visualize them

def build_lstm_attention(seq_len, n_features):
    inp = keras.Input(shape=(seq_len, n_features), name='sensor_window')

    # Layer 1: LSTM — builds temporal representations
    x = layers.LSTM(128, return_sequences=True,
                    kernel_regularizer=keras.regularizers.l2(L2))(inp)
    x = layers.Dropout(DROPOUT)(x)

    # Layer 2: LSTM — deeper temporal abstraction
    x = layers.LSTM(64, return_sequences=True,
                    kernel_regularizer=keras.regularizers.l2(L2))(x)
    x = layers.Dropout(DROPOUT)(x)

    # Attention: learns which of the 30 cycles matters most for RUL
    attended, _ = DotProductAttention()(x)

    # Collapse sequence dimension — average attended representations
    x = layers.GlobalAveragePooling1D()(attended)

    # Classification head (Block 1: Dense output for regression)
    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(L2))(x)
    x = layers.BatchNormalization()(x)   # Block 2: normalize to stabilize training
    out = layers.Dense(1, name='rul')(x) # 1 output neuron = regression (Block 1)

    return keras.Model(inputs=inp, outputs=out, name='LSTM_Attention_RUL')

model_attn = build_lstm_attention(SEQ_LEN, N_FEATURES)
model_attn.summary()
history_attn, pred_attn, rmse_attn, mae_attn = train_model(model_attn, 'lstm_model')
results['LSTM+Attention'] = dict(history=history_attn, rmse=rmse_attn, mae=mae_attn)

## Architecture Comparison — Presentation Slide

In [ ]:
# ── Comparison table ───────────────────────────────────────────────────────
CYAN = '#00d4ff'; AMBER = '#f59e0b'; NAVY = '#0a0f1e'

comparison = pd.DataFrame({
    'Architecture':  ['SimpleRNN', 'GRU', 'LSTM', 'LSTM + Attention'],
    'Class concept': [
        'Block 4: vanishing gradient baseline',
        'Block 4: reset+update gates, fewer params',
        'Block 4: cell state highway, 4 gates',
        'Block 4+1: attention over timesteps'
    ],
    'RMSE (cycles)': [results[k]['rmse'] for k in ['SimpleRNN', 'GRU', 'LSTM', 'LSTM+Attention']],
    'MAE  (cycles)': [results[k]['mae']  for k in ['SimpleRNN', 'GRU', 'LSTM', 'LSTM+Attention']],
})

print(comparison.to_string(index=False))

# ── Bar chart ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
archs = comparison['Architecture']
rmses = comparison['RMSE (cycles)']
colors = [AMBER if a == 'LSTM + Attention' else CYAN for a in archs]
bars = ax.bar(archs, rmses, color=colors, edgecolor='white', alpha=0.85)
ax.bar_label(bars, fmt='%.1f', color='white', fontsize=11)
ax.set_title('RMSE by Architecture — Class Curriculum Progression', fontsize=13)
ax.set_ylabel('Test RMSE (cycles)')
ax.set_facecolor(NAVY)
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.savefig('../data/processed/architecture_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best = comparison.loc[comparison['RMSE (cycles)'].idxmin(), 'Architecture']
print(f'\nBest model: {best}  →  saved as backend/model/lstm_model.keras')

In [ ]:
# ── Training curves for all 4 models ──────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
model_keys = ['SimpleRNN', 'GRU', 'LSTM', 'LSTM+Attention']

for ax, key in zip(axes, model_keys):
    h = results[key]['history'].history
    ax.plot(h['loss'],     label='Train', color=CYAN,  linewidth=1.5)
    ax.plot(h['val_loss'], label='Val',   color=AMBER, linewidth=1.5)
    ax.set_title(f'{key}\nRMSE={results[key]["rmse"]:.1f}', fontsize=10)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.set_facecolor(NAVY)

axes[0].set_ylabel('Huber Loss')
fig.suptitle('Training Curves — Loss Convergence per Architecture', fontsize=13)
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.savefig('../data/processed/training_history.png', dpi=150, bbox_inches='tight')
plt.show()